In [2]:
import torch
import numpy as np
import equinox as eqx
import jax
import jax.numpy as jnp
from jaxtyping import Array, Float, Int, PyTree  # https://github.com/google/jaxtyping
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import pandas as pd
import pickle
import os
from itertools import combinations
from tqdm import tqdm

# example of calculating the frechet inception distance
import numpy
from numpy import cov
from numpy import trace
from numpy import iscomplexobj
from numpy.random import random
from scipy.linalg import sqrtm
from scipy.stats import wasserstein_distance_nd


# import ot  # POT library

# def sinkhorn_wasserstein(x, y, reg=0.1):
#     n, m = x.shape[0], y.shape[0]
#     a, b = np.ones(n)/n, np.ones(m)/m
#     M = ot.dist(x, y)  # cost matrix
#     return ot.sinkhorn2(a, b, M, reg)

from main_project.train import train_classifier
from main_project.model import targetClassifier
from main_project.visualize import plot_mmd_image_heatmaps_full, plot_latent_dim_vs_average_mmd, plot_gamma_vs_mmd

from main_project.utils import load
from main_project.environment import MODELS_DIM, INTERMEDIATE_FRACTIONS, MAX_POINTS, GAMMA, LABELS


In [26]:
figure_3 = pd.read_csv("../../data/figure_3.csv").iloc[:, 0:]

In [27]:
figure_3

,latent_dim,0.0001,0.001,0.01,0.1,1.0
0,2,"MMD: 0.3363, W-Dist: 4.0658, Conf: 0.5327","MMD: 0.0838, W-Dist: 2.6385, Conf: 0.8253","MMD: 0.1192, W-Dist: 2.8560, Conf: 0.8085","MMD: 0.2508, W-Dist: 3.4012, Conf: 0.7628","MMD: 0.7149, W-Dist: 4.5156, Conf: 0.5947"
1,8,"MMD: 0.0175, W-Dist: 4.3333, Conf: 0.9036","MMD: 0.0011, W-Dist: 4.1095, Conf: 0.9525","MMD: 0.0159, W-Dist: 4.4143, Conf: 0.9692","MMD: 0.2693, W-Dist: 5.3497, Conf: 0.9944","MMD: 1.1058, W-Dist: 6.9842, Conf: 1.0000"
2,16,"MMD: 0.0013, W-Dist: 4.3809, Conf: 0.9539","MMD: 0.0018, W-Dist: 4.4694, Conf: 0.9592","MMD: 0.0333, W-Dist: 4.8799, Conf: 0.9781","MMD: 0.5830, W-Dist: 6.3282, Conf: 0.9991","MMD: 1.1657, W-Dist: 7.4006, Conf: 1.0000"
3,32,"MMD: 0.0084, W-Dist: 4.4021, Conf: 0.9406","MMD: 0.0009, W-Dist: 4.3015, Conf: 0.9626","MMD: 0.0115, W-Dist: 4.5383, Conf: 0.9781","MMD: 0.2802, W-Dist: 5.3839, Conf: 0.9985","MMD: 1.0433, W-Dist: 6.9550, Conf: 1.0000"


In [ ]:
summary_df = pd.read_csv("../../data/evaluation_summary.csv")



In [5]:
summary_df.shape

(900, 9)

In [6]:


summary_df.head(20)

,latent_dim,gamma,source_label,target_label,mmd_latent,wasserstein_distance_latent,mmd_image,classifier_confidence_image,wasserstein_distance
0,2,0.0001,0,1,0.182518,1.054852,0.222846,0.691300,3.485699
1,2,0.0001,0,2,0.810024,3.652645,0.342624,0.806450,4.400009
2,2,0.0001,0,3,0.374284,1.789761,0.196718,0.483856,3.446245
3,2,0.0001,0,4,0.325214,1.596033,0.575616,0.293658,5.561946
4,2,0.0001,0,5,0.334752,1.545212,0.468250,0.347349,5.021378
5,2,0.0001,0,6,0.288299,1.647427,0.321956,0.604955,4.291954
6,2,0.0001,0,7,0.172173,1.372922,0.264079,0.608955,4.159952
7,2,0.0001,0,8,0.656955,1.983116,0.765976,0.152111,6.420708
8,2,0.0001,0,9,0.700108,2.025387,0.728053,0.251622,5.928025
9,2,0.0001,1,2,0.330044,2.171024,0.203039,0.785726,3.644497
